<a href="https://colab.research.google.com/github/jrhumberto/2026-mestrado-pep/blob/main/etl_siconfi.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Notebook para extração de bases - Dimensão Fiscal

>Trata-se de notebook para extração de dados provenientes das eleições do SICONFI e do SIDRA(IBGE)

  
  **Responsável/Ano**: Humberto Bezerra de M. Júnior - 2026

Observação: Agradecimentos aos idealizadores dos Pacotes aqui utilizados:


*   siconfir
*   sidrar



## Fase 0 - Etapa Inicial: Instalação de Pacotes globais

In [9]:
if (!require(pacman)) install.packages("pacman")
pacman::p_load("sidrar","siconfi","httr", "jsonlite", "stringr", "dplyr",
               "tidyr", "knitr", "devtools", "ggplot2", "officer", "tibble" )


devtools::install_github("tchiluanda/rsiconfi")
devtools::install_github("davidgohel/flextable")

library(rlang)
library(sidrar)
library(rsiconfi)
library(dplyr)
library(purrr)
library(tidyr)


Loading required package: pacman

Installing package into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)

Warning message:
“package ‘siconfi’ is not available for this version of R

A version of this package for your version of R might be available elsewhere,
see the ideas at
https://cran.r-project.org/doc/manuals/r-patched/R-admin.html#Installing-packages”
Warning message:
“'BiocManager' not available.  Could not check Bioconductor.

Please use `install.packages('BiocManager')` and then retry.”
Warning message in p_install(package, character.only = TRUE, ...):
“”
Warning message in library(package, lib.loc = lib.loc, character.only = TRUE, logical.return = TRUE, :
“there is no package called ‘siconfi’”
Warning message in pacman::p_load("sidrar", "siconfi", "httr", "jsonlite", "stringr", :
“Failed to install/load:
siconfi”
Skipping install of 'rsiconfi' from a github remote, the SHA1 (cb105807) has not changed since last install.
  Use `force = TRUE` to force installation

S

## Fase 1 - Extração: Obtenção de 6 variáveis do SICONFI e 1 do SIDRA(IBGE)


1. **Dívida consolidada líquida** ( *dcl* ) — valor absoluto da dívida menos disponibilidades (recursos em curto prazo disponíveis em caixa/estoque do governo). Proveniente do Anexo 2 do RGF.
1. **População do estado no nno** ( *pop* ) - trata-se da população que foi informada pelo Ente no RGF daquele ano.
1. **PIB a preços correntes** ( *pib_prc* ) - trata-se de PIB de todas as receitas e riquezas apuradas pelo ente naquele ano. Dado extraído do SIDRA/IBGE. **IMPORTANTE**: **valor do ano 2024 está ainda indisponível em 2026**. Por essa falta, foi atribuído valor 0.
1. **Receita Corrente Líquida** ( *rcl* ) - trata-se de valor absoluto da Receita Corrente Líquida do ente.
1. **Receita Corrente Líquida Ajustada** ( rcla ) - trata-se do valor de  Receita Corrente Líquida reduzido do montante de Transferências Obrigatórias da União Relativas às Emendas Individuais (art. 166-A, §1º, da CF).
1. **Despesa total com pessoal** ( *dtp* ) - Somatório de gastos com pessoal.
1. **Percentual da Despesa Total com Pessoal em relação a sua RCLA** ( *percentual_dtp_rcla* ) - Razão entre a despesa total com pessoal (dtp) em relação ao valor da Receita Corrente Líquida Ajustada (rcla) em percentual. Proxy de rigidez fiscal embasado em norma (art. 20 e 22, da LRF), que estabelece governadores não podem comprometer mais de 49%







In [10]:
anos <- 2019:2024

estados <- tibble(
  uf = c("RO","AC","AM","RR","PA","AP","TO",
         "MA","PI","CE","RN","PB","PE","AL","SE","BA",
         "MG","ES","RJ","SP","PR","SC","RS",
         "MS","MT","GO","DF"),
  cod_ibge = c("11","12","13","14","15","16","17",
               "21","22","23","24","25","26","27","28","29",
               "31","32","33","35","41","42","43",
               "50","51","52","53")
)

# Create the 'data' directory if it doesn't exist
dir.create("data", showWarnings = FALSE)

# ── Função por estado × ano ───────────────────────────────────────────────────

extrair_rgf <- function(uf, cod_ibge, ano) {

  message(sprintf(">>> %s | %d", uf, ano))

  anexo1 <- tryCatch(
    get_rgf(
      year        = ano,
      periodicity = "Q",
      period      = 3,
      report_tp   = 1,
      annex       = "01",
      entity      = cod_ibge,
      co_power    = "E"
    ),
    error = function(e) {
      warning(sprintf("Extração do Anexo 01! Erro em %s/%d: %s", uf, ano, e$message))
      return(NULL)
    }
  )


  anexo2 <- tryCatch(
    get_rgf(
      year        = ano,
      periodicity = "Q",
      period      = 3,
      report_tp   = 1,
      annex       = "02",
      entity      = cod_ibge,
      co_power    = "E"
    ),
    error = function(e) {
      warning(sprintf("Extração do Anexo 02! Erro em %s/%d: %s", uf, ano, e$message))
      return(NULL)
    }
  )

  linha_vazia <- tibble(
    uf = uf, ano = ano, dcl = NA_real_,
    pop = NA_real_, pib_prec_corr = NA_real_, rcl = NA_real_,
    rcla = NA_real_,dtp = NA_real_, dtp_perc_rcla = NA_real_
  )

  if (is.null(anexo1) || nrow(anexo1) == 0) return(linha_vazia)

  if (is.null(anexo2) || nrow(anexo2) == 0) return(linha_vazia)

  # ── Extração: filtra só por cod_conta + coluna ─────────────────────

  dcl <- anexo2 %>%
        filter(
          cod_conta == "DividaConsolidadaLiquida",
          coluna    == "Até o 3º Quadrimestre"
        ) %>% pull(valor) %>% as.numeric() %>% first()

  pop <- anexo1 %>%
    slice(1) %>%
    pull(populacao) %>%
    as.numeric()

  # PIB a preços correntes (Mil Reais) — tabela 5938, variável 37
  pib <- tryCatch(
    get_sidra(
    x          = 5938,
    variable   = 37,
    period     = c(as.character(ano)),
    geo        = "State",
    geo.filter = list("State" = as.character(cod_ibge)),
    format     = 3),
    error = function(e) NULL
  )

  # Transforma PIP a preços correntes * 1000
  pib_prc <- if (is.null(pib) || nrow(pib) == 0 || !("Valor" %in% names(pib))) { 0 } else {
    as.numeric(pib$Valor) * 1000
  }


  rcl <- anexo1 %>%
    filter(
      cod_conta == "ReceitaCorrenteLiquidaLimiteLegal",
      coluna    == "Valor"
    ) %>%
    pull(valor) %>% as.numeric() %>% first()

  rcla <- anexo1 %>%
    filter(
      cod_conta == "ReceitaCorrenteLiquidaAjustada",
      coluna    == "Valor"
    ) %>%
    pull(valor) %>% as.numeric() %>% first()

  dtp <- anexo1 %>%
    filter(
      cod_conta == "DespesaComPessoalTotal",
      coluna    == "Valor"
    ) %>%
    pull(valor) %>% as.numeric() %>% first()

  percentual_dtp_rcla <- anexo1 %>%
    filter(
      cod_conta == "DespesaComPessoalTotal",
      coluna    == "% sobre a RCL Ajustada"
    ) %>%
    pull(valor) %>% as.numeric() %>% first()

  tibble(
    uf            = uf,
    ano           = ano,
    dcl           = dcl,
    pop           = pop,
    pib_prc       = pib_prc,
    rcl           = rcl,
    rcla          = rcla,
    dtp           = dtp,
    percentual_dtp_rcla = percentual_dtp_rcla
  )
}

# ── Criar a Grade (UF por cada ano i, i+1...) ──────────────────────────────────────

grade <- crossing(estados, ano = anos)   # 162 combinações = 27 UF x (2019 até 2024)

# ── Iteração ──────────────────────────────────────────────────────────────────

df_rgf <- pmap_dfr( list(uf = grade$uf, cod_ibge = grade$cod_ibge, ano = grade$ano),extrair_rgf)


# ── Ordenação ─────────────────────────────────────────────────────────────────

df_rgf <-  df_rgf %>% arrange(uf, ano)

# ── Verificação ───────────────────────────────────────────────────────────────

cat("NA em Colunas:\n")
print(colSums(is.na(df_rgf)))
cat("\n\nLinhas:", nrow(df_rgf), "\n")          # Esperado: 162
print(names(df_rgf))

# ── Exportar ──────────────────────────────────────────────────────────────────
write.csv(df_rgf, "./data/fiscal_uf_2019_2024.csv", row.names = FALSE)

>>> AC | 2019

Joining with `by = join_by(esfera)`
Joining with `by = join_by(esfera)`
Considering all categories once 'classific' was set to 'all' (default)

>>> AC | 2020

Joining with `by = join_by(esfera)`
Joining with `by = join_by(esfera)`
Considering all categories once 'classific' was set to 'all' (default)

>>> AC | 2021

Joining with `by = join_by(esfera)`
Joining with `by = join_by(esfera)`
Considering all categories once 'classific' was set to 'all' (default)

>>> AC | 2022

Joining with `by = join_by(esfera)`
Joining with `by = join_by(esfera)`
Considering all categories once 'classific' was set to 'all' (default)

>>> AC | 2023

Joining with `by = join_by(esfera)`
Joining with `by = join_by(esfera)`
Considering all categories once 'classific' was set to 'all' (default)

>>> AC | 2024

Joining with `by = join_by(esfera)`
Joining with `by = join_by(esfera)`
Considering all categories once 'classific' was set to 'all' (default)

>>> AL | 2019

Joining with `by = join_by(esfe

NA em Colunas:
                 uf                 ano                 dcl                 pop 
                  0                   0                   0                   0 
            pib_prc                 rcl                rcla                 dtp 
                  0                   0                   0                   0 
percentual_dtp_rcla 
                  0 


Linhas: 162 
[1] "uf"                  "ano"                 "dcl"                
[4] "pop"                 "pib_prc"             "rcl"                
[7] "rcla"                "dtp"                 "percentual_dtp_rcla"


## Fase 2 - Etapa Opcional: Criação de tabela(s) para dissertação

### Funções Genéricas: monta_tabela

In [11]:
library(flextable)
library(officer)
library(dplyr)
library(tibble)

monta_tabela <- function(dados, largs, titulo, fonte, path_my, name) {

  # Cria diretorio
  dir.create(path_my, showWarnings = FALSE)

  if ("uf" %in% names(dados) && "ano" %in% names(dados)) {
      dados <- dados %>% arrange(uf, ano)
  } else if ("ano" %in% names(dados)) {
      dados <- dados %>% arrange(ano)
  }

  t <- flextable(dados) %>%
    set_header_labels(uf = "UF", ano= "Ano")  %>%
    border_remove() %>%
    hline_top(part = "header", border = fp_border(color="black", width = 1.5)) %>%
    hline_bottom(part = "header", border = fp_border(color="black", width = 1)) %>%
    hline_bottom(part = "body", border = fp_border(color="black", width = 1.5)) %>%

    # Legenda (Título) em tamanho 12 e Fonte
    set_caption(caption = titulo, style = "Table Caption") %>%
    add_footer_lines(fonte) %>%

    # Fonte e tamanho 10
    font(fontname = "Times New Roman", part = "all") %>%
    fontsize(size = 10, part = "body") %>%
    fontsize(size = 10, part = "header") %>%
    fontsize(size = 10, part = "footer") %>%

    # Fixa larguras das colunas e acerta a borda
    width(j = seq_along(largs), width = largs) %>%
    fix_border_issues()


  # Guarde
  set_flextable_defaults(big.mark = "", decimal.mark =",",
         font.family ='Times New Roman', font.size=10 )


  t <- t %>%  colformat_int(j = 'ano', big.mark = "")

  if ("dtp_perc_rcla" %in% t$col_keys) {
        t <- t %>% colformat_double(
          j            = "dtp_perc_rcla",
          big.mark     = "",
          decimal.mark = ",",
          digits       = 2,
          suffix       = "%"
        )
  }

  save_as_docx(t, path = paste0(path_my,'/', name, ".docx"))

}


Attaching package: ‘flextable’


The following object is masked from ‘package:purrr’:

    compose




### Montagem de Tabelas com 2 grupos de Variáveis

In [12]:
# Grupo1 - VAriáveis economicas financeiras
# Grupo2 - Variáveis de responsabilidade fiscal

df_rgf4 <- df_rgf %>% select(uf, ano, dcl, pop, pib_prc, rcl)
w_rgf4 <- c(0.5, 0.5, 1.1, 1.1, 1.1, 1.1)

df_rgf5 <- df_rgf %>% select(uf, ano,dtp, rcla, percentual_dtp_rcla)
w_rgf5 <- c(0.4, 0.5, 1.5, 1.5, 1.5)


monta_tabela(
  df_rgf4, w_rgf4,
  "Tabela 4 – Variáveis financeiras e econômicas: despesa corrente líquida, população, PIB a preços correntes, receita corrente líquida - UF do Brasil (2019 até 2024)",
  "Fonte: Elaboração própria a partir de dados do SICONFI (BRASIL, 2024) e do SIDRA (IBGE, 2022)",
  'tabelas', 't4_tabela_fiscal'
)

monta_tabela(
  df_rgf5, w_rgf5,
  "Tabela 5 – Variáveis de responsabilidade fiscal: despesa total com pessoal, receita corrente líquida ajustada e percentual entre despesa total com pessoal e receita corrente líquida ajustada - UF do Brasil (2019 até 2024)",
  "Fonte: Elaboração própria a partir de dados do SICONFI (BRASIL, 2024)",
  'tabelas', 't5_tabela_fiscal'
)

---
